# SPATIAL INTELLIGENCE — PART 3
## Homework 02 — ETM

## 1. Import the needed libraries

In [92]:
from topologicpy.Vertex import Vertex
from topologicpy.Edge import Edge
from topologicpy.Wire import Wire
from topologicpy.Face import Face
from topologicpy.Shell import Shell
from topologicpy.Cell import Cell
from topologicpy.CellComplex import CellComplex
from topologicpy.Cluster import Cluster
from topologicpy.Topology import Topology
from topologicpy.Dictionary import Dictionary
from topologicpy.Helper import Helper
from topologicpy.Grid import Grid
from topologicpy.Graph import Graph
from topologicpy.Color import Color

## 2. Check the TopologicPy Version

In [93]:
print("This notebook requires topologicpy version 0.9.18 or newer.")
print(Helper.Version())

This notebook requires topologicpy version 0.9.18 or newer.
The version that you are using (0.9.29) is EQUAL TO the latest version available on PyPI.


## 3. Set your renderer
* Visual Studio Code: `"vscode"`
* Google Colab: `"colab"`
* Browser: `"browser"`

In [94]:
renderer = "vscode"

## 4. Utility functions

In [95]:
def reset_dictionaries(shell):
    faces = Topology.Faces(shell)
    for i, f in enumerate(faces):
        d = Topology.Dictionary(f)
        keys = Dictionary.Keys(d)
        for key in keys:
            if not key == 'face_id':
                d = Dictionary.RemoveKey(d, key)
        f = Topology.SetDictionary(f, d)

def transfer_dicts_by_key(topologies, selectors, key):
    dicts = {}
    for t in topologies:
        d = Topology.Dictionary(t)
        value = Dictionary.ValueAtKey(d, key, None)
        if value:
            dicts[str(value)] = t
    for s in selectors:
        d = Topology.Dictionary(s)
        value = Dictionary.ValueAtKey(d, key, None)
        if value:
            f = dicts.get(str(value), None)
            if f:
                f = Topology.SetDictionary(f, d)

## 5. Load the floor plan outline (OBJ)

In [96]:
OBJ_PATH = r"C:\Users\etmaglari\IAAC\etmaglari_gML\Homework02\homework02_outline.obj"
objects = Topology.ByOBJPath(OBJ_PATH)
print(f"Imported {len(objects)} objects")

floor_face = None
for obj in objects:
    faces = Topology.Faces(obj)
    if faces:
        floor_face = faces[0]
        break
    wires = Topology.Wires(obj)
    if wires:
        floor_face = Face.ByWire(wires[0])
        if floor_face:
            break

print("Floor face loaded:", floor_face is not None)

b_r = Wire.BoundingRectangle(floor_face)
d_br = Topology.Dictionary(b_r)
xmin   = Dictionary.ValueAtKey(d_br, "xmin")
xmax   = Dictionary.ValueAtKey(d_br, "xmax")
ymin   = Dictionary.ValueAtKey(d_br, "ymin")
ymax   = Dictionary.ValueAtKey(d_br, "ymax")
width  = Dictionary.ValueAtKey(d_br, "width")
length = Dictionary.ValueAtKey(d_br, "length")
print(f"Bounds: x=[{xmin:.1f}, {xmax:.1f}]  y=[{ymin:.1f}, {ymax:.1f}]")
print(f"Size: {width:.1f} x {length:.1f} units")

Imported 2 objects
Floor face loaded: True
Bounds: x=[0.0, 30.5]  y=[-3.7, 12.2]
Size: 30.5 x 15.8 units


In [97]:
# Strip interior holes — use only the outer boundary for isovist
ext_wire = Topology.ExternalBoundary(floor_face)
iso_face = Face.ByWire(ext_wire)
n_verts = len(Topology.Vertices(ext_wire))
print(f"External boundary vertices: {n_verts}")
print(f"Clean isovist face valid: {iso_face is not None}")

External boundary vertices: 136
Clean isovist face valid: True


## 6. Show the floor plan

In [98]:
Topology.Show(floor_face,
              camera=[0, 0, 6],
              faceColor=[210, 210, 250],
              faceOpacity=1,
              edgeColor="white",
              edgeWidth=3,
              showVertices=False,
              backgroundColor="black",
              width=800, height=600,
              renderer=renderer)

## 7. Create two grids
* `grid1` — coarse vertex grid for isovist viewpoints
* `grid2` — dense edge grid for slicing the floor plan

In [99]:
COARSE_STEP = 28
DENSE_STEP  = 1
uRange1 = list(range(0, int(width)  + COARSE_STEP, COARSE_STEP))
vRange1 = list(range(0, int(length) + COARSE_STEP, COARSE_STEP))
uRange2 = list(range(0, int(width)  + DENSE_STEP,  DENSE_STEP))
vRange2 = list(range(0, int(length) + DENSE_STEP,  DENSE_STEP))
grid1 = Grid.VerticesByDistances(floor_face, clip=True, uRange=uRange1, vRange=vRange1)
grid2 = Grid.EdgesByDistances(floor_face,    clip=True, uRange=uRange2, vRange=vRange2)

In [100]:
Topology.Show(floor_face, grid2,
              camera=[0, 0, 6],
              faceColor=[210, 210, 250],
              faceOpacity=1,
              edgeColor="grey",
              edgeWidth=2,
              showVertices=False,
              backgroundColor="black",
              width=800, height=600,
              renderer=renderer)

## 8. Slice the floor plan with the dense grid

In [101]:
shell = Topology.Slice(floor_face, grid2)
faces = Topology.Faces(shell)
for i, f in enumerate(faces):
    d = Dictionary.ByKeyValue("face_id", "face_"+str(i+1))
    f = Topology.SetDictionary(f, d)
print(f"Grid cells: {len(faces)}")

Grid cells: 347


## 9. Derive the analysis graph and isovist viewpoints

In [102]:
analysis_graph = Graph.ByTopology(shell)
g_verts   = Graph.Vertices(analysis_graph)

# Sample every Nth graph vertex — all guaranteed inside the floor plan
SAMPLE_STEP = 8
iso_verts = g_verts[::SAMPLE_STEP]
print(f"Analysis graph vertices: {len(g_verts)}")
print(f"Isovist viewpoints: {len(iso_verts)}")

Analysis graph vertices: 347
Isovist viewpoints: 44


## 10. Compute isovists
* Only viewpoints inside the floor boundary produce valid isovists.
* Time-consuming — expect a few minutes.

In [103]:
isovists = []
for v in iso_verts:
    iso = Face.Isovist(iso_face, v)
    isovists.append(iso)
valid_isovists = [i for i in isovists if i]
print(f"Valid isovists: {len(valid_isovists)}")

Face.RemoveCollinearEdges - Error: The input face parameter is not a valid face. Returning None.
caller name: Isovist
Face.RemoveCollinearEdges - Error: The input face parameter is not a valid face. Returning None.
caller name: Isovist
Topology.SelfMerge - Error: The input topology is not a valid topology. Returning None
Shell.ByFaces - Error: Could not create shell. Returning None.
Shell.ExternalBoundary - Error: External boundary could not be found. Returning None.
Face.Isovist - Error: Could not create isovist. Returning None.
Face.RemoveCollinearEdges - Error: The input face parameter is not a valid face. Returning None.
caller name: Isovist
Face.RemoveCollinearEdges - Error: The input face parameter is not a valid face. Returning None.
caller name: Isovist
Face.RemoveCollinearEdges - Error: The input face parameter is not a valid face. Returning None.
caller name: Isovist
Face.RemoveCollinearEdges - Error: The input face parameter is not a valid face. Returning None.
caller name: 

In [104]:
Topology.Show(iso_face, valid_isovists,
              faceOpacity=0.6,
              showEdges=False,
              camera=[0, 0, 6],
              backgroundColor="black",
              width=800, height=600,
              renderer=renderer)

## 11. Compute visibility count at each isovist viewpoint
Count how many dense graph vertices fall inside each isovist.

In [105]:
new_verts = []
n_list = []
for i, iso in enumerate(isovists):
    if iso:
        v = iso_verts[i]
        b_list = Vertex.IsInternal2D(g_verts, iso)
        b_list = [b for b in b_list if b]
        n = len(b_list)
        n_list.append(n)
        d = Dictionary.ByKeyValue("visibility", n)
        v = Topology.SetDictionary(v, d)
        new_verts.append(v)

Face.ExternalBoundary - Error: The input face parameter is not a topologic face. Returning None.
Face.ExternalBoundary - Error: The input face parameter is not a topologic face. Returning None.
Face.ExternalBoundary - Error: The input face parameter is not a topologic face. Returning None.
Face.ExternalBoundary - Error: The input face parameter is not a topologic face. Returning None.
Face.ExternalBoundary - Error: The input face parameter is not a topologic face. Returning None.
Face.ExternalBoundary - Error: The input face parameter is not a topologic face. Returning None.
Face.ExternalBoundary - Error: The input face parameter is not a topologic face. Returning None.
Face.ExternalBoundary - Error: The input face parameter is not a topologic face. Returning None.
Face.ExternalBoundary - Error: The input face parameter is not a topologic face. Returning None.
Face.ExternalBoundary - Error: The input face parameter is not a topologic face. Returning None.
Face.ExternalBoundary - Error:

## 12. Interpolate visibility to dense graph vertices and show

In [106]:
for v in g_verts:
    new_v = Vertex.InterpolateValue(v, vertices=new_verts, n=2, key="visibility")

In [107]:
minValue = min(n_list)
maxValue = max(n_list)
for v in g_verts:
    d = Topology.Dictionary(v)
    vb = Dictionary.ValueAtKey(d, "visibility")
    color = Color.AnyToHex(Color.ByValueInRange(vb, minValue=minValue, maxValue=maxValue, colorScale="thermal"))
    d = Dictionary.SetValueAtKey(d, "vb_color", color)
    d = Dictionary.SetValueAtKey(d, "size", 16)
    v = Topology.SetDictionary(v, d)

In [108]:
reset_dictionaries(shell)
_ = transfer_dicts_by_key(faces, g_verts, "face_id")

In [109]:
Topology.Show(faces,
              faceColorKey="vb_color",
              faceOpacity=1,
              showEdges=False,
              showVertices=False,
              camera=[0, 0, 6],
              backgroundColor="black",
              width=800, height=600,
              renderer=renderer)